<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Roteiro de Atividade Prática</h2>

<br>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Projeto Análise Exploratória de Dados  
**Tema da Semana:** Projeto de Análise Exploratória de Dados: Planejamento e Exploração Inicial  
**Semana 6**  
**Aula 4:** Registro das descobertas iniciais

<br>


# Integração das evidências e recomendação inicial

**Situação profissional**

Nas três aulas anteriores, vocês definiram a pergunta do projeto, planejaram a investigação e executaram a exploração inicial.

> **Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?**

Como **assistentes de dados da equipe comercial**, agora vocês precisam transformar os resultados obtidos em um registro útil para decisão: interpretar evidências, explicar diferenças entre níveis de análise, aprofundar a frente de **Resultado** e formular uma recomendação inicial acompanhada de limites e próximos passos.

**Missão da aula**

> **Analisar e integrar as evidências do projeto para formular uma recomendação inicial de prioridade, explicitar seus limites e definir o próximo passo da investigação.**

## Passo a passo

- Preparem os arquivos e recuperem as evidências produzidas na Aula 3.
- Analisem como os níveis de desconto se relacionam com aproveitamento, margem bruta e lucro bruto.
- Expliquem o que as análises global e por grupos respondem sobre horas de equipe e aproveitamento.
- Organizem as evidências das frentes de Atração, Conversão e Resultado.
- Avaliem qual aspecto merece prioridade para uma primeira ação de melhoria e justifiquem a escolha com evidências.
- Registrem os limites da recomendação e uma análise adicional necessária.
- Troquem o registro com outra dupla, façam os ajustes necessários ou justifiquem sua manutenção e salvem o notebook preenchido.


## Preparação

Arquivos usados nesta atividade:

- `DADOS3EMC3B1S6A4_professor.ipynb`
- `rede_papelarias.csv`

1. Salvem os dois arquivos **na mesma pasta**.
2. Abram o arquivo `.ipynb` no **Visual Studio Code**.
3. Se solicitado, selecionem um **kernel Python** disponível.
4. Executem a célula abaixo.

O código reproduz as decisões de preparação das aulas anteriores: remove somente a duplicata integral, preserva os registros com `espera_media_min` ausente e recria os indicadores `cupons_por_100_entradas` e `margem_bruta_pct`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

arquivo_csv = Path("rede_papelarias.csv")

if not arquivo_csv.is_file():
    raise FileNotFoundError(
        "Arquivo 'rede_papelarias.csv' não encontrado. "
        "Salve o CSV na mesma pasta do notebook ou faça o upload para a sessão."
    )

dados = pd.read_csv(arquivo_csv)
dados_trabalho = dados.drop_duplicates().copy()

if dados_trabalho["entradas"].isna().any() or (dados_trabalho["entradas"] <= 0).any():
    raise ValueError("Há valores ausentes ou não positivos em 'entradas'.")

if dados_trabalho["receita_liquida"].isna().any() or (dados_trabalho["receita_liquida"] <= 0).any():
    raise ValueError("Há valores ausentes ou não positivos em 'receita_liquida'.")

dados_trabalho["cupons_por_100_entradas"] = (
    dados_trabalho["cupons_emitidos"] / dados_trabalho["entradas"] * 100
)

dados_trabalho["margem_bruta_pct"] = (
    dados_trabalho["lucro_bruto"] / dados_trabalho["receita_liquida"] * 100
)

ordem_lojas = sorted(
    dados_trabalho["loja"].dropna().unique(),
    key=lambda x: int(str(x).replace("L", ""))
)

def correlacao(df, x, y):
    pares = df[[x, y]].dropna()
    return len(pares), pares[x].corr(pares[y])

print(f"Registros após remover a duplicata integral: {len(dados_trabalho):,}".replace(",", "."))
print(f"Valores ausentes de espera_media_min: {dados_trabalho['espera_media_min'].isna().sum()}")


## Etapa 1 — Recuperem os registros da A3
**Tempo de referência: 3 minutos**

A célula abaixo recalcula um quadro compacto com evidências das frentes de **Atração** e **Conversão** e recupera os quatro níveis examinados para `horas_equipe × cupons_por_100_entradas`.

Não refaçam a atividade da A3. Usem este quadro apenas como ponto de partida para a análise de hoje.


In [ ]:
# Evidências principais da A3
evidencias_a3 = []

for frente, relacao, x, y in [
    ("Atração", "Investimento em divulgação × entradas",
     "investimento_divulgacao", "entradas"),
    ("Conversão", "Disponibilidade × aproveitamento",
     "disponibilidade_produtos_pct", "cupons_por_100_entradas"),
    ("Conversão", "Espera × aproveitamento",
     "espera_media_min", "cupons_por_100_entradas"),
    ("Conversão", "Horas de equipe × aproveitamento — global",
     "horas_equipe", "cupons_por_100_entradas"),
]:
    n, r = correlacao(dados_trabalho, x, y)
    evidencias_a3.append({
        "frente": frente,
        "relação": relacao,
        "n": n,
        "r": r
    })

display(pd.DataFrame(evidencias_a3).round(3))

# Quatro níveis de horas de equipe × aproveitamento
_, r_global_horas = correlacao(
    dados_trabalho, "horas_equipe", "cupons_por_100_entradas"
)

rs_loja_todos = []
for loja in ordem_lojas:
    rec = dados_trabalho[dados_trabalho["loja"] == loja]
    _, r = correlacao(rec, "horas_equipe", "cupons_por_100_entradas")
    rs_loja_todos.append(r)

dados_faixa = dados_trabalho[dados_trabalho["entradas"].between(150, 199)].copy()
_, r_faixa_rede = correlacao(
    dados_faixa, "horas_equipe", "cupons_por_100_entradas"
)

rs_loja_faixa = []
for loja in ordem_lojas:
    rec = dados_faixa[dados_faixa["loja"] == loja]
    _, r = correlacao(rec, "horas_equipe", "cupons_por_100_entradas")
    rs_loja_faixa.append(r)

quatro_niveis = pd.DataFrame([
    {
        "nível de análise": "Todas as lojas | todos os dias",
        "resultado": f"r = {r_global_horas:+.3f}"
    },
    {
        "nível de análise": "Cada loja | todos os dias",
        "resultado": f"r entre {min(rs_loja_todos):+.3f} e {max(rs_loja_todos):+.3f}"
    },
    {
        "nível de análise": "Todas as lojas | 150–199 entradas",
        "resultado": f"r = {r_faixa_rede:+.3f}"
    },
    {
        "nível de análise": "Cada loja | 150–199 entradas",
        "resultado": f"r entre {min(rs_loja_faixa):+.3f} e {max(rs_loja_faixa):+.3f}"
    },
])

display(quatro_niveis)


### Registro 1 — organizem o ponto de partida

**1. Consultem as evidências de Atração e Conversão registradas no encerramento da A3 e confiram sua correspondência com o quadro recuperado. Usem esses registros na integração final; não é necessário transcrevê-los novamente.**


## Etapa 2 — Aprofundem a frente de Resultado
**Tempo de referência: 12 minutos**

A frente ainda pendente pergunta:

> **Como os níveis de desconto se relacionam com o aproveitamento do movimento, o lucro bruto e a margem bruta das lojas?**

Primeiro, executem a célula abaixo. Ela apresenta:

- a correlação global entre desconto e cada resultado;
- um resumo por nível de desconto;
- boxplots para comparar as distribuições entre os níveis.

Lembrem-se: diferentes níveis de desconto são **condições observadas nos dados**. As análises mostram associação e diferenças entre grupos, não provam que o desconto causou os resultados.


In [ ]:
# Relações globais com desconto
relacoes_desconto = []
for y, rotulo in [
    ("cupons_por_100_entradas", "Aproveitamento"),
    ("margem_bruta_pct", "Margem bruta (%)"),
    ("lucro_bruto", "Lucro bruto (R$)")
]:
    n, r = correlacao(dados_trabalho, "desconto_campanha_pct", y)
    relacoes_desconto.append({
        "relação": f"Desconto × {rotulo}",
        "n": n,
        "r": r
    })

print("Relações globais")
display(pd.DataFrame(relacoes_desconto).round(3))

# Resumo por nível de desconto
resumo_desconto = (
    dados_trabalho
    .groupby("desconto_campanha_pct")
    .agg(
        n=("lucro_bruto", "size"),
        mediana_aproveitamento=("cupons_por_100_entradas", "median"),
        mediana_margem=("margem_bruta_pct", "median"),
        mediana_lucro_bruto=("lucro_bruto", "median"),
    )
)

print("Resumo por nível de desconto")
display(resumo_desconto.round(2))

niveis = sorted(dados_trabalho["desconto_campanha_pct"].dropna().unique())

def boxplot_por_desconto(coluna, titulo, ylabel):
    grupos = [
        dados_trabalho.loc[
            dados_trabalho["desconto_campanha_pct"] == nivel, coluna
        ].dropna()
        for nivel in niveis
    ]
    plt.figure(figsize=(7, 4.5))
    plt.boxplot(grupos)
    plt.xticks(range(1, len(niveis) + 1), [f"{int(n)}%" for n in niveis])
    plt.xlabel("Desconto da campanha")
    plt.ylabel(ylabel)
    plt.title(titulo)
    plt.grid(axis="y", alpha=0.15)
    plt.tight_layout()
    plt.show()

boxplot_por_desconto(
    "cupons_por_100_entradas",
    "Aproveitamento por nível de desconto",
    "Cupons por 100 entradas"
)

boxplot_por_desconto(
    "margem_bruta_pct",
    "Margem bruta por nível de desconto",
    "Margem bruta (%)"
)

boxplot_por_desconto(
    "lucro_bruto",
    "Lucro bruto por nível de desconto",
    "Lucro bruto (R$)"
)


### Registro 2 — interpretem a frente de Resultado

**2. Expliquem como o aproveitamento e a margem bruta se comportam entre os níveis de desconto. Que benefício observado e que redução de margem precisam ser considerados conjuntamente?**

**Resposta:**


**3. A correlação global entre desconto e lucro bruto é próxima de zero. Expliquem por que esse resultado, sozinho, não encerra a investigação. Usem a tabela por níveis e os boxplots para sustentar a resposta.**

**Resposta:**


**4. Os dados permitem identificar qual nível apresentou a maior mediana de lucro bruto no período observado. Isso é suficiente para recomendá-lo como o desconto que maximizará o lucro da rede? Expliquem.**

**Resposta:**


## Etapa 3 — Expliquem o contraste entre o global e os grupos
**Tempo de referência: 8 minutos**

Na A3, vocês **executaram e registraram** quatro leituras de `horas_equipe × aproveitamento`. Agora o objetivo muda: vocês precisam **explicar** o que cada leitura responde.

Retomem o quadro dos quatro níveis exibido no início do notebook.


### Registro 3 — perguntas diferentes, informações complementares

**5. Expliquem por que os quatro resultados de `horas_equipe × aproveitamento` não são contraditórios. Para cada nível, indiquem qual pergunta ele ajuda a responder.**

**Resposta:**


**6. Avaliem o que essa investigação acrescenta a uma decisão para a rede. Por que uma análise adicional por loja e por faixa de movimento pode revelar oportunidades que o resultado global sozinho não mostra?**

**Resposta:**


**7. Registrem uma recomendação de aprofundamento para o relatório: o que deveria ser analisado por loja e por faixa de movimento e que tipo de fator não registrado no conjunto também pode estar relacionado às diferenças?**

**Resposta:**


## Etapa 4 — Integrem as três frentes e formulem a recomendação inicial
**Tempo de referência: 14 minutos**

Agora organizem o que o projeto produziu.

Antes de responder, executem a célula abaixo. Ela compara a **consistência da direção** de algumas relações dentro das lojas. O objetivo não é criar uma nova frente, mas apoiar a avaliação das evidências já obtidas.


In [ ]:
# Consistência das relações dentro das lojas
def intervalo_r_por_loja(x, y):
    valores = []
    for loja in ordem_lojas:
        rec = dados_trabalho[dados_trabalho["loja"] == loja]
        _, r = correlacao(rec, x, y)
        valores.append(r)
    return min(valores), max(valores)

linhas_consistencia = []
for aspecto, relacao, x, y in [
    ("Divulgação", "investimento × entradas",
     "investimento_divulgacao", "entradas"),
    ("Disponibilidade", "disponibilidade × aproveitamento",
     "disponibilidade_produtos_pct", "cupons_por_100_entradas"),
    ("Espera", "espera × aproveitamento",
     "espera_media_min", "cupons_por_100_entradas"),
    ("Horas de equipe", "horas × aproveitamento",
     "horas_equipe", "cupons_por_100_entradas"),
]:
    _, r_global = correlacao(dados_trabalho, x, y)
    r_min, r_max = intervalo_r_por_loja(x, y)
    linhas_consistencia.append({
        "aspecto": aspecto,
        "relação examinada": relacao,
        "r global": r_global,
        "menor r por loja": r_min,
        "maior r por loja": r_max
    })

display(pd.DataFrame(linhas_consistencia).set_index("aspecto").round(3))


### Registro 4 — quadro de acompanhamento do projeto

**8. Organizem uma evidência principal de cada frente. Para cada uma, registrem uma interpretação e um limite.**

**Atração**
- Evidência:
- Interpretação:
- Limite:

**Conversão**
- Evidência:
- Interpretação:
- Limite:

**Resultado**
- Evidência:
- Interpretação:
- Limite:


**9. Avaliem qual aspecto merece prioridade para uma primeira ação de melhoria. Justifiquem com pelo menos duas evidências do projeto e registrem pelo menos um limite da recomendação.**

**Prioridade recomendada:**


**Justificativa:**


**Limite:**


**10. Retomem o aprofundamento registrado na questão 7 e ajustem-no, se necessário, à recomendação da questão 9. Indiquem o próximo passo sem repetir integralmente a resposta anterior.**

**Próximo passo:**


**11. Formulem uma nova pergunta que possa orientar esse aprofundamento.**

**Nova pergunta:**


## Etapa 5 — Troquem, revisem e entreguem
**Tempo de referência: 6 minutos**

Troquem o Registro 4 com outra dupla.

A dupla que recebe deve verificar três pontos:

1. a recomendação está sustentada por evidências visuais e/ou numéricas?
2. existe pelo menos um limite explícito?
3. o próximo passo decorre das evidências encontradas?

**12. Registrem uma melhoria feita após a devolutiva da outra dupla. Se nenhum ajuste for necessário, justifiquem com base nos três critérios de revisão.**

**Melhoria realizada ou justificativa para manter o registro:**


Salvem o notebook preenchido após a revisão.


## Critérios para avaliação da atividade

- Organiza evidências das três frentes da investigação.
- Explica os resultados da frente de Resultado usando correlações, tabelas e distribuições.
- Explica por que análises globais e por grupos podem responder a perguntas diferentes.
- Integra evidências visuais e numéricas ao formular uma recomendação inicial.
- Avalia limites da recomendação e evita atribuição causal indevida.
- Propõe um aprofundamento coerente com os resultados observados.
- Formula uma nova pergunta relacionada às evidências e aos limites identificados.
- Revisa o registro após a troca com outra dupla.

**Entrega:** um notebook preenchido e salvo por dupla ao final da aula, contendo o registro de acompanhamento do projeto: pergunta, evidências, interpretações, limites, recomendação inicial e próximo passo.


---

**Sobre os dados**

O conjunto `rede_papelarias.csv` foi elaborado para fins didáticos e simula registros de operação diária de uma pequena rede de lojas. As conclusões descrevem somente esse conjunto e não devem ser generalizadas para empresas reais.

**Limites metodológicos**

As análises identificam associações e diferenças entre grupos. Elas não demonstram causa e efeito, não determinam número ideal de funcionários, não permitem calcular retorno sobre investimento sem informações adicionais e não estabelecem um desconto ótimo para a rede.


# Expectativa de respostas

## Registro 1 — evidências recuperadas

**1. Referências para conferência, sem nova transcrição**
- **Atração:** investimento em divulgação × entradas, `r ≈ +0,418`.
- **Conversão:** disponibilidade × aproveitamento, `r ≈ +0,400`.
- **Conversão:** espera × aproveitamento, `r ≈ −0,647`.
- Também pode ser retomada a relação horas de equipe × aproveitamento, desde que a dupla preserve o nível de análise indicado.

## Registro 2 — frente de Resultado

**2.** O aproveitamento aumenta entre os níveis de desconto, enquanto a margem bruta diminui. O benefício observado é maior aproveitamento, acompanhado de menor margem bruta: níveis mais altos aparecem associados a mais cupons por 100 entradas, mas a uma parcela menor de lucro bruto em relação à receita líquida.

Valores de referência:
- desconto × aproveitamento: `r ≈ +0,299`;
- desconto × margem bruta: `r ≈ −0,740`.

**3.** Desconto × lucro bruto apresenta `r ≈ 0,000`. Isso não significa que o desconto seja irrelevante. As medianas de lucro bruto variam entre os níveis e o comportamento não é monotônico:
- 0%: ≈ R$ 1.130;
- 5%: ≈ R$ 1.138;
- 10%: ≈ R$ 1.186;
- 15%: ≈ R$ 1.173;
- 20%: ≈ R$ 1.095.

Os boxplots também apresentam ampla sobreposição. O coeficiente global próximo de zero resume uma relação linear global e não substitui a comparação entre os níveis.

**4.** Não. A tabela permite identificar o nível com maior mediana de lucro bruto nos registros observados; isso não demonstra qual desconto maximizará o lucro em uma intervenção futura. As condições não foram atribuídas de modo a isolar o efeito do desconto, e outras características podem diferir entre os grupos. O lucro bruto já desconta o custo das mercadorias vendidas; faltam informações sobre despesas operacionais e custos de implementação para avaliar o resultado final da decisão. Tamanhos diferentes e sobreposição das distribuições devem ser considerados na interpretação, mas não impedem automaticamente a comparação nem são, isoladamente, a razão para rejeitar um desconto ótimo.

## Registro 3 — global × grupos

**5.** Explicação esperada:
- **todas as lojas e todos os dias (`r ≈ −0,434`)**: responde à relação observada no conjunto completo da rede;
- **cada loja com todos os dias (`r` ≈ −0,005 a +0,089)**: responde à relação dentro de cada unidade considerando todo o movimento;
- **todas as lojas com 150–199 entradas (`r ≈ +0,052`)**: responde à relação na rede quando o nível de movimento é aproximado, ainda misturando lojas;
- **cada loja com 150–199 entradas (`r` ≈ +0,126 a +0,466)**: responde à relação dentro de cada unidade em dias de movimento semelhante.

Os resultados não são contraditórios porque cada nível responde a uma pergunta diferente.

**6.** A investigação mostra que o agregado pode ocultar comportamentos específicos de classes. Uma política única para toda a rede pode perder oportunidades de ajuste por contexto. Aprofundar por loja e faixa de movimento permite procurar padrões recorrentes e diferenças específicas antes de propor medidas operacionais.

**7.** Espera-se recomendação de analisar, por loja e faixas de entradas, combinações de condições operacionais associadas a melhor aproveitamento e resultado comercial. A resposta deve mencionar que fatores não registrados — por exemplo, organização da equipe, perfil do público, mix de produtos, eventos locais ou características do atendimento — também podem estar relacionados às diferenças. Esses fatores são hipóteses para coleta futura, não conclusões do conjunto atual.

## Registro 4 — integração e recomendação

**8. Exemplos de organização**
- **Atração:** investimento × entradas (`r ≈ +0,418`); interpretação: maior investimento aparece associado a maior movimento; limite: não há cálculo de retorno nem prova causal.
- **Conversão:** espera × aproveitamento (`r ≈ −0,647`); interpretação: maior espera aparece associada a menor aproveitamento; limite: a relação não informa como reduzir a espera nem prova causalidade.
- **Resultado:** níveis maiores de desconto aparecem associados a maior aproveitamento e menor margem, enquanto lucro bruto não apresenta relação linear global clara; limite: não há evidência para declarar desconto ótimo.

**9. Exemplo de recomendação fundamentada**
Uma recomendação possível é **priorizar a investigação de alternativas para reduzir a espera e avaliar sua viabilidade em um teste operacional**. O conjunto não demonstra que essa prioridade seja superior às demais em benefício operacional ou econômico.

Evidências que podem fundamentar esse exemplo:
- espera × aproveitamento tem a associação linear mais intensa entre as condições operacionais examinadas: `r ≈ −0,647`;
- a direção negativa aparece nas oito lojas quando analisadas separadamente (`r` por loja aproximadamente de `−0,598` a `−0,244`);
- a mudança da associação de horas de equipe entre recortes justifica investigar ajustes contextualizados; não desqualifica essa frente nem demonstra o efeito de alterar a carga de trabalho;
- desconto apresenta trade-off entre aproveitamento e margem e não mostra ganho linear de lucro bruto;
- divulgação se relaciona ao movimento, mas o conjunto não permite avaliar retorno sobre o investimento.

Limites obrigatórios: associação não demonstra causalidade; maior intensidade de correlação não mede o benefício de uma intervenção; a forma de reduzir a espera não está determinada; custos operacionais e de implementação e outros fatores relevantes não estão disponíveis; a recomendação se restringe a este conjunto simulado.

Outras prioridades podem atender plenamente à atividade quando integrarem evidências pertinentes, reconhecerem os resultados que limitam a proposta e delimitarem o que ainda precisa ser verificado. Não avaliar a resposta pela coincidência com o exemplo de prioridade apresentado.

**10.** A dupla deve retomar o aprofundamento da questão 7 e conectá-lo à prioridade recomendada. Pode mantê-lo ou ajustá-lo, explicitando a ligação com a recomendação, sem repetir integralmente o texto anterior.

**11. Exemplo de nova pergunta**
> **Em quais lojas e faixas de movimento diferentes condições operacionais estão associadas a melhor aproveitamento e resultado comercial, e quais fatores ainda não registrados podem ajudar a explicar essas diferenças?**

Aceitar formulações equivalentes que decorram das evidências e mantenham o caráter investigativo.

## Registro 5 — devolutiva

**12.** Aceitar melhoria concreta provocada pela revisão: inclusão de evidência, correção de interpretação causal, explicitação de limite, ajuste da recomendação ou refinamento da nova pergunta. Aceitar também “nenhum ajuste necessário”, desde que a justificativa mostre o atendimento aos três critérios: sustentação em evidências, limite explícito e próximo passo decorrente dos resultados.

## Critério geral

A resposta deve organizar evidência, interpretação, limite e próximo passo. Não aceitar afirmações causais, “desconto ótimo”, número ideal de funcionários ou generalizações para outras redes.
